## Imports and helper functions

In [ ]:
import glob
import pprint
import struct
import sys
from array import array
from collections.abc import Iterable
from dataclasses import dataclass, field
from enum import IntEnum
from pathlib import Path
from typing import BinaryIO

# Please install PIL or Pillow:
from PIL.ImagePalette import ImagePalette
from PIL import Image, ImageSequence

from IPython.display import display

In [ ]:
# This directory contains one subdirectory for each game:
# - jetpack
# - xjetpack
# - shareware_jetpack
GAMESDIR = Path("/home/denilson/Games/")

## Jetpack classes

In [ ]:
# Use this module without installing it.
sys.path.insert(0, "./src")
from jetpyck.level import *

## JetpackLevelPack

In [ ]:
def ALL_JETLEV_DAT():
    return sorted(GAMESDIR.glob("*/JETLEV.DAT", case_sensitive=False))

In [ ]:
for filename in ALL_JETLEV_DAT():
    print(filename)
    with open(filename, "rb") as f:
        pack = JetpackLevelPack.unpack(f)
    for i, lvl in enumerate(pack.levels):
        print("{:>3}: {}".format(i + 1, lvl.description.decode("ascii")))

In [ ]:
lvl.filename

In [ ]:
for filename in ALL_JETLEV_DAT():
    print(filename)
    pack = JetpackLevelPack.load_from_bytes(filename.read_bytes())
    for i, lvl in enumerate(pack.levels):
        print(lvl.as_printable_text())

In [ ]:
lvl.filename

In [ ]:
for filename in ALL_JETLEV_DAT():
    print(filename)
    pack = JetpackLevelPack.load_from_filename(filename)
    for i, lvl in enumerate(pack.levels):
        print(lvl.as_printable_half_blocks())

In [ ]:
lvl.filename

In [ ]:
for filename in ALL_JETLEV_DAT():
    print(filename)
    pack = JetpackLevelPack.load_from_filename(filename)
    for i, lvl in enumerate(pack.levels):
        print(lvl.as_printable_sextant_blocks())

In [ ]:
for filename in ALL_JETLEV_DAT():
    print(filename)
    pack = JetpackLevelPack.load_from_filename(filename)
    for i, lvl in enumerate(pack.levels):
        print(lvl.as_printable_octant_blocks())

In [ ]:
for filename in ALL_JETLEV_DAT():
    print(filename)
    pack = JetpackLevelPack.load_from_filename(filename)
    for i, lvl in enumerate(pack.levels):
        print(lvl.as_printable_braille())

## JetpackLevel

In [ ]:
def ALL_LEVELS_JET():
    return sorted(GAMESDIR.glob("*/LEVELS/*.JET", case_sensitive=False))

In [ ]:
for filename in ALL_LEVELS_JET():
    print(filename)
    with open(filename, "rb") as f:
        lvl = JetpackLevel.unpack(f)
    print(lvl.as_printable_text())

In [ ]:
for filename in ALL_LEVELS_JET():
    print(filename)
    lvl = JetpackLevel.load_from_bytes(filename.read_bytes())
    print(lvl.as_printable_half_blocks())

In [ ]:
for filename in ALL_LEVELS_JET():
    print(filename)
    lvl = JetpackLevel.load_from_filename(filename)
    print(lvl.as_printable_sextant_blocks())

### Blank level

This level was saved completely empty in the level editor.

All the fields are zero. Including the player and door positions.

In [ ]:
blank = JetpackLevel.load_from_filename(GAMESDIR / 'jetpack/LEVELS/BLANK.JET')
blank

## Tilemap classes (and functions) (i.e. gfx-related)

In [ ]:
class JetpackEditorTileset:
    def __init__(self, filename):
        self.filename = filename
        self.image = Image.open(self.filename)

    def __repr__(self) -> str:
        return 'JetpackEditorTileset({!r})'.format(self.filename)

    def _get_tile_box(self, id: int) -> tuple[int]:
        if not 0 <= id < 120:
            raise ValueError('Invalid tile {!r}'.format(id))

        tiles_per_row = 20
        x = id % tiles_per_row
        y = id // tiles_per_row
        w = h = 12
        dx = dy = 16

        px = 2 + x * dx
        py = 96 + y * dy
        return (px, py, px + w, py + h)

    def get_tile(self, id: int) -> Image.Image:
        box = self._get_tile_box(id)
        return self.image.crop(box)

    def get_tiles(self, id: int) -> list[Image.Image]:
        box = self._get_tile_box(id)
        return ImageSequence.all_frames(self.image, lambda image: image.crop(box))

In [ ]:
def image_from_level(level: JetpackLevel, tileset: JetpackEditorTileset) -> Image.Image:
    w = h = 12
    output = Image.new(tileset.image.mode, (w * level.tilemap.width, h * level.tilemap.height))
    for (x, y, tile) in level.tilemap.items():
        output.paste(tileset.get_tile(tile), (x * w, y * h))

    # TODO:
    # - Animation frames
    # - Add enemies
    # - Add door and player
    # - Separate layers
    # - Proper background/foreground, including transparency
    return output

## Exploration

In [ ]:
foo = JetpackLevel()
pprint.pprint(foo)

In [ ]:
pathname = Path('./levelpacks/xmasjetpack/XMAS0.JET')
with open(pathname, 'rb') as f:
    foo = JetpackLevel.unpack(f, pathname.name)
    pprint.pprint(foo)

In [ ]:
list(foo.tilemap.rows())

In [ ]:
list(foo.tilemap.items())

In [ ]:
with Image.open('./Editor-tileset.webp') as tileset:
    print(tileset.n_frames)

In [ ]:
tileset

In [ ]:

tileset.crop((2, 96, 2+12, 96+12))

In [ ]:
tileset = JetpackEditorTileset('./Editor-tileset.webp')
tileset

In [ ]:
for tile in range(0, 120):
    display(tileset.get_tile(tile))

In [ ]:
for im in tileset.get_tiles(2):
    display(im)

In [ ]:
image_from_level(foo, tileset)